# Claude Agent sample using Amazon Bedrock models
* **Host:** AgentCore Runtime
* **Instrumentation:** Claude Native
* **Observability:** CloudWatch

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Claude Agents SDK
* Docker/Finch running
* Amazon CloudWatch Access
* Enable [transaction search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) on Amazon CloudWatch. 


In [ ]:
!pip install --force-reinstall -U -r requirements-dev.txt --quiet
!pip install --force-reinstall -U -r requirements.txt --quiet

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [ ]:
%%writefile claude_agent.py
"""Sample Claude Agent with Bedrock"""

from claude_agent_sdk import (
    ClaudeAgentOptions,
    query,
    tool,
    create_sdk_mcp_server,
    ResultMessage,
)
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Init server
app = BedrockAgentCoreApp()


# Create a custom tool
@tool("add", "Add two numbers", {"a": float, "b": float})
async def add(args):
    """add two numbers"""
    result = args["a"] + args["b"]
    return {"content": [{"type": "text", "text": f"Result: {result}"}]}

# Create an SDK MCP server
calculator = create_sdk_mcp_server(
    name="calculator",
    version="1.0.0",
    tools=[add]
)

# Use it with Claude
options = ClaudeAgentOptions(
    system_prompt="You are a helpful assistant. Always use tools if available for the operation. Even if it is simple.",
    mcp_servers={"calc": calculator},
    allowed_tools=["mcp__calc__add"],
    output_format="json"
)


async def invoke_agent(user_query):
    """Invoke Claude Agent"""
    async for message in query(
        prompt=user_query,
        options=options,
    ):
        if isinstance(message, ResultMessage):
            yield message.result


@app.entrypoint
async def claude_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_query = payload.get("prompt")
    print("User input:", user_query)

    response = ""
    async for result in invoke_agent(user_query):
        response += result
    return response

if __name__ == "__main__":
    app.run()


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

When configuring for containerized environment (such as docker) add the following command, an example is given below:

`CMD ["opentelemetry-instrument", "python", "claude_agent.py"]`


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "ac_runtime_claude_agent_sdk_obsy_demo"
response = agentcore_runtime.configure(
    entrypoint="claude_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    disable_otel=True
)
response

### Launching agent to AgentCore Runtime
Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

**How telemetry reaches CloudWatch on AgentCore Runtime.** When `AGENT_OBSERVABILITY_ENABLED=true`, the runtime auto-instruments the Python process with the AWS Distro for OpenTelemetry (ADOT) SDK and exports **collector-less, directly to the CloudWatch/X-Ray OTLP endpoints using SigV4**. There is **no local OTLP collector** inside the container — a startup diagnostic confirmed nothing listens on `localhost:4318` or `:4317`. Because of this:

- **Do not** set `OTEL_EXPORTER_OTLP_ENDPOINT`, `OTEL_EXPORTER_OTLP_PROTOCOL`, or the `OTEL_*_EXPORTER` selectors here. The platform already points ADOT at CloudWatch; overriding the endpoint (e.g. to `http://localhost:4318`) redirects export to a collector that doesn't exist and produces `Connection refused .../v1/logs` errors.
- We keep only the flags that add value on top of the managed pipeline:
  - `CLAUDE_CODE_ENABLE_TELEMETRY=1` and `CLAUDE_CODE_ENHANCED_TELEMETRY_BETA=1` — turn on Claude Code's native telemetry / beta traces.
  - `OTEL_LOG_USER_PROMPTS`, `OTEL_LOG_TOOL_DETAILS`, `OTEL_LOG_TOOL_CONTENT` — opt in to prompt/tool content, which is redacted by default. Omit these if you don't want content captured.

**Caveat on the native Claude Code stream.** Claude Code emits its own OTLP but has no SigV4 signer, and there is no local collector to receive it, so on Runtime its native spans have no reachable CloudWatch target. The traces you see in the GenAI Observability dashboard come from the platform's ADOT auto-instrumentation of the Python process, not from Claude Code's own exporter. Capturing Claude Code's native spans in CloudWatch would require a SigV4-capable collector (for example an OTel Collector with the `sigv4auth` extension) that you run and point Claude Code at.

See the [Claude Code monitoring docs](https://code.claude.com/docs/en/monitoring-usage) and the [CloudWatch OTLP endpoints docs](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/CloudWatch-OTLPEndpoint.html). Content was rephrased for compliance with licensing restrictions.

In [ ]:
%%writefile .env
# Application configurations
AWS_REGION=<region>
CLAUDE_CODE_USE_BEDROCK=1
ANTHROPIC_MODEL=us.anthropic.claude-sonnet-4-5-20250929-v1:0

# ADOT configurations
AGENT_OBSERVABILITY_ENABLED=true
AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true

# Claude monitoring configurations
CLAUDE_CODE_ENABLE_TELEMETRY=1
CLAUDE_CODE_ENHANCED_TELEMETRY_BETA=1
OTEL_LOG_USER_PROMPTS=1
OTEL_LOG_TOOL_DETAILS=1
OTEL_LOG_TOOL_CONTENT=1

# Claude trace export configurations
OTEL_TRACES_EXPORTER=otlp
OTEL_METRICS_EXPORTER=none
OTEL_LOGS_EXPORTER=none
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_EXPORTER_OTLP_ENDPOINT=<OTLP Endpoint>
OTEL_EXPORTER_OTLP_HEADERS=<OTLP header>

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# On AgentCore Runtime, the platform auto-instruments the Python process with the
# ADOT SDK (collector-less) and sends telemetry directly to CloudWatch with SigV4.
# Do NOT set OTEL_EXPORTER_OTLP_ENDPOINT / _PROTOCOL / _*_EXPORTER here: there is no
# local collector (diagnostics showed nothing listening on localhost:4318), and
# overriding the endpoint redirects ADOT away from CloudWatch and breaks export.
#
# We only enable Claude Code's native telemetry flags and opt in to content.
!agentcore launch \
    --env CLAUDE_CODE_USE_BEDROCK=1 \
    --env AWS_REGION=us-east-1 \
    --env ANTHROPIC_MODEL=us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
    --env CLAUDE_CODE_ENABLE_TELEMETRY=1 \
    --env CLAUDE_CODE_ENHANCED_TELEMETRY_BETA=1 \
    --env OTEL_LOG_USER_PROMPTS=1 \
    --env OTEL_LOG_TOOL_DETAILS=1 \
    --env OTEL_LOG_TOOL_CONTENT=1 \
    --env OTEL_TRACES_EXPORTER=otlp \
    --env OTEL_METRICS_EXPORTER=none \
    --env OTEL_LOGS_EXPORTER=none \
    --env OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf \
    --env OTEL_EXPORTER_OTLP_ENDPOINT=$OTEL_EXPORTER_OTLP_ENDPOINT \
    --env OTEL_EXPORTER_OTLP_HEADERS=$OTEL_EXPORTER_OTLP_HEADERS

    # --env AGENT_OBSERVABILITY_ENABLED=true \
    # --env AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true \

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
!agentcore status


### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
!agentcore invoke '{"prompt": "what is 2+23?"}'

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run

# Congratulations!